# Moving features and the best subspace

Section 7.2 of the notes.

A DeepONet writes every output in one subspace, so its error is at least the
error of the best subspace, which the singular values of the outputs give. This
notebook takes the same profile at many random positions and computes that
error: the sharper the profile, the more dimensions a fixed subspace needs.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

## 1. Shifted profiles

On a periodic grid of $256$ points, the profile $\tanh(c\cos\pi(x-a))$ at $1000$
random positions $a$. For small $c$ it is a smooth wave; for large $c$ it is a
plateau at $\pm1$ with two sharp interfaces.

In [ ]:
def shifted_profiles(x, c, n_shifts, rng):
    # Columns: tanh(c cos(pi (x - a))) for n_shifts random positions a.
    a = rng.uniform(-1, 1, n_shifts)
    return np.tanh(c * np.cos(np.pi * (x[:, None] - a[None, :])))


def floor(Y):
    # E_q = (sum_{i>q} s_i^2 / sum_i s_i^2)^(1/2) for q = 0, 1, ..., min(d, J).
    s2 = np.linalg.svd(Y, compute_uv=False) ** 2
    tail = np.concatenate([np.cumsum(s2[::-1])[::-1], [0.0]])
    return np.sqrt(tail / s2.sum())


x = -1 + 2 * np.arange(256) / 256
rng = np.random.default_rng(0)
widths = [0.5, 3, 30]
profiles = {c: shifted_profiles(x, c, 1000, rng) for c in widths}
floors = {c: floor(Y) for c, Y in profiles.items()}
for c in widths:
    q = int(np.argmax(floors[c] < 1e-2))
    print(f"c = {c:3}: E_q < 0.01 from q = {q}")

In [ ]:
plt.rcParams.update(
    {
        "font.family": "STIXGeneral",
        "mathtext.fontset": "stix",
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
colors = {0.5: "#2a78d6", 3: "#1baf7a", 30: "#eb6834"}
fig, axes = plt.subplots(1, 2, figsize=(3.8, 1.3), gridspec_kw=dict(wspace=0.4))
for c in widths:
    for a in (-0.3, 0.2, 0.7):
        axes[0].plot(
            x,
            np.tanh(c * np.cos(np.pi * (x - a))),
            color=colors[c],
            lw=1.0,
            label=f"$c = {c}$" if a == -0.3 else None,
        )
    q = np.arange(len(floors[c]))
    axes[1].semilogy(
        q[:101], np.maximum(floors[c][:101], 1e-16), color=colors[c], lw=1.3
    )
axes[0].set(xlabel="$x$", title="three positions of each profile", ylim=(-1.15, 1.15))
axes[1].set(
    xlabel="dimension $q$",
    ylabel="$E_q$",
    title="error of the best subspace",
    ylim=(1e-4, 1.5),
    xlim=(0, 100),
)
fig.legend(
    *axes[0].get_legend_handles_labels(),
    loc="lower center",
    bbox_to_anchor=(0.5, 0.98),
    ncol=3,
    frameon=False,
)
fig.savefig(FIGURES / "moving_features.pdf", bbox_inches="tight")
plt.show()

## 2. A fixed basis oscillates at a moving step

The data matrix $\bm Y$ has the sharp profile at the $1000$ random positions as
columns. Its first left singular vectors are vectors of grid values: plotted
against $x$ they are basis functions, close to waves. A profile at a new position,
written as its best combination of the first $q$ of them (its projection),
overshoots and oscillates next to each interface.

In [ ]:
def basis(Y, q):
    # The first q left singular vectors of Y, as columns.
    return np.linalg.svd(Y, full_matrices=False)[0][:, :q]


c, a_new, q = 30, 0.13, 16
Y = profiles[c]
V = basis(Y, q)
step = np.tanh(c * np.cos(np.pi * (x - a_new)))
fig, axes = plt.subplots(1, 3, figsize=(5.2, 1.3), gridspec_kw=dict(wspace=0.35))
ax = axes[0]
for a in (-0.6, -0.1, 0.45):
    ax.plot(x, np.tanh(c * np.cos(np.pi * (x - a))), lw=1.0)
ax.set(
    xlabel="$x$",
    title="3 of the 1000 columns of $\\mathbf{Y}$",
    xticks=[-1, 0, 1],
    ylim=(-1.2, 1.2),
)
ax = axes[1]
for k in range(4):
    ax.plot(x, V[:, k] * np.sqrt(len(x)), lw=1.0, label=f"$\\mathbf{{v}}_{k + 1}$")
ax.set(
    xlabel="$x$",
    title="basis vectors $\\mathbf{v}_1,\\dots,\\mathbf{v}_4$",
    xticks=[-1, 0, 1],
)
ax = axes[2]
ax.plot(x, step, color="black", lw=1.6, label="step at a new position")
ax.plot(x, V @ (V.T @ step), color="#2a78d6", lw=1.0, label=f"best combination of {q}")
ax.set(
    xlabel="$x$",
    title=f"best combination, $q = {q}$",
    xticks=[-1, 0, 1],
    ylim=(-1.35, 1.35),
)
fig.legend(
    *ax.get_legend_handles_labels(),
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    ncol=2,
    frameon=False,
    fontsize=7,
    handlelength=1.6,
)
fig.savefig(FIGURES / "basis_oscillation.pdf", bbox_inches="tight")
plt.show()